# 02 — Fetch Movie Metadata from OMDb API

This notebook queries the [OMDb API](https://www.omdbapi.com/) to retrieve metadata for each downloaded script: IMDb rating, Rotten Tomatoes score, genre, director, plot summary, etc.

**Run this notebook after** `01_scrape_scripts.ipynb`.

**Requires:** `OMDB_API_KEY` set in `.env` (free tier limit: 1,000 requests/day)

**Outputs:**
- `movie_metadata_temp.csv` — checkpoint file updated after each daily run
- `movie_metadata_clean.csv` — final clean file, created once all movies are processed

## 1. Setup — List Scripts and Define Helper Functions

Load environment variables, list all downloaded scripts, define `normalizar_titulo()` for filename-to-title conversion, and run a quick test query to verify the API key works.

In [ ]:
import os
#pip install python-dotenv
from dotenv import load_dotenv
load_dotenv()
import re
import requests
from thefuzz import process

# List all downloaded scripts
scripts_folder = "scripts"
files = [f for f in os.listdir(scripts_folder) if f.endswith('.txt')]

def normalize_title(filename):
    # Remove extension and replace underscores/dashes with spaces
    title = os.path.splitext(filename)[0]
    title = title.replace('_', ' ').replace('-', ' ')
    # Remove special characters and extra whitespace
    title = re.sub(r'[^a-zA-Z0-9 ]', '', title)
    title = re.sub(r'\s+', ' ', title).strip()
    return title.lower()

# Normalize all filenames to readable titles
normalized_titles = [normalize_title(f) for f in files]

print(f"Total scripts downloaded: {len(normalized_titles)}")
print("Example normalized titles:")
for t in normalized_titles[:10]:
    print(f"- {t}")

# Define single-movie query function (used for quick tests below)
def fetch_omdb_rating(title, api_key):
    url = f"http://www.omdbapi.com/?t={title}&apikey={api_key}"
    res = requests.get(url)
    data = res.json()
    if data.get('Response') == 'True':
        return {
            'Title': data.get('Title'),
            'Year': data.get('Year'),
            'imdbRating': data.get('imdbRating'),
            'RottenTomatoes': next((r['Value'] for r in data.get('Ratings', []) if r['Source'] == 'Rotten Tomatoes'), None)
        }
    return None

api_key = os.getenv('OMDB_API_KEY')
if not api_key:
    raise ValueError("OMDB_API_KEY environment variable not found. Please set it before running this code.")

# Quick sanity check — query first 5 titles
results = []
for t in normalized_titles[:5]:
    print(f"\nQuerying: {t}")
    info = fetch_omdb_rating(t, api_key)
    print(info)
    results.append({'normalized_title': t, 'omdb_info': info})


## 2. Debug — Test API Connection

Isolate a specific title to confirm the request URL format and API key authorization are correct before running the bulk query.

In [ ]:
# DEBUG CELL - Investigate "Black Snake Moan"
import time
import pandas as pd
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

print("="*70)
print("DEBUG: Searching for 'Black Snake Moan'")
print("="*70)

# 1. Find the file
matching_files = [f for f in files if 'black' in f.lower() and 'snake' in f.lower()]
print(f"\nFiles found with 'black snake': {matching_files}")

# 2. Normalize
if matching_files:
    title_norm = normalize_title(matching_files[0])
    print(f"Normalized title: '{title_norm}'")
else:
    title_norm = "black snake moan"
    print(f"Using manual title: '{title_norm}'")

# 3. Attempt a direct query
print(f"\nAttempting OMDb query...")
print(f"URL: http://www.omdbapi.com/?t={title_norm}&apikey=***")

try:
    url = f"http://www.omdbapi.com/?t={title_norm}&apikey={api_key}"
    res = requests.get(url, timeout=5)
    res.raise_for_status()
    data = res.json()
    
    print(f"\nStatus Code: {res.status_code}")
    print(f"Response: {data.get('Response')}")
    print(f"Title: {data.get('Title')}")
    print(f"Year: {data.get('Year')}")
    print(f"Error (if any): {data.get('Error')}")
    
    if data.get('Response') == 'True':
        print("\n✅ FOUND")
        print(f"   Title: {data.get('Title')}")
        print(f"   Year: {data.get('Year')}")
        print(f"   IMDb Rating: {data.get('imdbRating')}")
        print(f"   Rotten Tomatoes: {next((r['Value'] for r in data.get('Ratings', []) if r['Source'] == 'Rotten Tomatoes'), 'N/A')}")
    else:
        print(f"\n❌ NOT FOUND: {data.get('Error')}")
        
except Exception as e:
    print(f"\n❌ QUERY ERROR: {type(e).__name__}")
    print(f"   Details: {str(e)}")

print("\n" + "="*70)


## 3. Bulk Query with Daily Checkpoint

Queries OMDb for all scripts, respecting the 1,000 requests/day free tier limit. Progress is saved to `movie_metadata_temp.csv` after each run so the process can be resumed the next day.

**Run this cell once per day** until all movies are processed.

In [ ]:
# CELL 3: Query ALL scripts with CHECKPOINT
import pandas as pd
import time
from datetime import datetime
import json
import os

print("="*70)
print("QUERYING ALL SCRIPTS FROM OMDB (With automatic resume)")
print("="*70)
# ─────────────────────────────────────────────────────────────────────
# 1. VERIFY AND LOAD CHECKPOINT (to resume progress)
# ─────────────────────────────────────────────────────────────────────
CHECKPOINT_FILE = "omdb_checkpoint.json"
RESULT_CSV = "movie_metadata_temp.csv"
MAX_REQUESTS_PER_DAY = 1000

# Function to load checkpoint
def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {
        'last_date': None,
        'queries_today': 0,
        'last_index': -1,
        'queried_indices': []
    }

# Function to save checkpoint
def save_checkpoint(checkpoint):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump(checkpoint, f, indent=2)

# Load checkpoint
checkpoint = load_checkpoint()
today = datetime.now().strftime("%Y-%m-%d")

# Reset counter if it's a new day
if checkpoint['last_date'] != today:
    print(f"\n🔄 New day detected: {today}")
    checkpoint['last_date'] = today
    checkpoint['queries_today'] = 0
    print(f"   Request counter reset: 0/{MAX_REQUESTS_PER_DAY}")
else:
    print(f"\n Resuming from: {today}")
    print(f"   Requests used today: {checkpoint['queries_today']}/{MAX_REQUESTS_PER_DAY}")
    available_requests = MAX_REQUESTS_PER_DAY - checkpoint['queries_today']
    print(f"   Available requests: {available_requests}")

# ─────────────────────────────────────────────────────────────────────
# 2. LOAD PREVIOUS RESULTS (if they exist)
# ─────────────────────────────────────────────────────────────────────

if os.path.exists(RESULT_CSV):
    df_prev_results = pd.read_csv(RESULT_CSV)
    print(f"\n✅ Previous results found: {len(df_prev_results)} movies already queried")
    already_queried = set(df_prev_results.index)
else:
    df_prev_results = None
    already_queried = set()
    print(f"\n📝 Starting queries from scratch")

# ─────────────────────────────────────────────────────────────────────
# 3. FUNCTION TO QUERY OMDB
# ─────────────────────────────────────────────────────────────────────

def fetch_omdb_rating(title, api_key, timeout=3):
    try:
        url = f"http://www.omdbapi.com/?t={title}&apikey={api_key}"
        res = requests.get(url, timeout=timeout)
        res.raise_for_status()
        data = res.json()
        
        if data.get('Response') == 'True':
            return {
                'Title': data.get('Title'),
                'Year': data.get('Year'),
                'imdbRating': data.get('imdbRating'),
                'RottenTomatoes': next((r['Value'] for r in data.get('Ratings', []) if r['Source'] == 'Rotten Tomatoes'), 'N/A'),
                'Plot': data.get('Plot'),
                'Director': data.get('Director'),
                'Actors': data.get('Actors'),
                'Genre': data.get('Genre'),
                'Type': data.get('Type'),
                'Response': 'Found'
            }
        else:
            return {
                'Title': None,
                'Year': None,
                'imdbRating': None,
                'RottenTomatoes': None,
                'Plot': None,
                'Director': None,
                'Actors': None,
                'Genre': None,
                'Type': None,
                'Response': data.get('Error', 'Not Found')
            }
    except Exception as e:
        return {
            'Title': None,
            'Year': None,
            'imdbRating': None,
            'RottenTomatoes': None,
            'Plot': None,
            'Director': None,
            'Actors': None,
            'Genre': None,
            'Type': None,
            'Response': f'Error: {str(e)}'
        }

# ─────────────────────────────────────────────────────────────────────
# 4. IDENTIFY PENDING MOVIES
# ─────────────────────────────────────────────────────────────────────

total_scripts = len(normalized_titles)
pending_movies = [i for i in range(total_scripts) if i not in already_queried]

print(f"\n📊 Current status:")
print(f"   Total movies: {total_scripts}")
print(f"   Already queried: {len(already_queried)}")
print(f"   Pending: {len(pending_movies)}")
print(f"   Available today: {MAX_REQUESTS_PER_DAY - checkpoint['queries_today']}")

# ─────────────────────────────────────────────────────────────────────
# 5. QUERY PENDING MOVIES (respecting daily limit)
# ─────────────────────────────────────────────────────────────────────

new_results = []
available_requests = MAX_REQUESTS_PER_DAY - checkpoint['queries_today']

if available_requests <= 0:
    print(f"\n⚠️  DAILY LIMIT REACHED (1000 requests)")
    print(f"   Come back tomorrow to continue with {len(pending_movies)} remaining movies")
else:
    # Limit queries to those available today
    movies_to_query = pending_movies[:available_requests]
    
    print(f"\n Querying {len(movies_to_query)} movies today...")
    print(f"   (Out of {len(pending_movies)} pending)")
    
    start_time = datetime.now()
    
    for idx, movie_idx in enumerate(movies_to_query):
        title_norm = normalized_titles[movie_idx]
        original_file = files[movie_idx]
        
        # Log progress
        if (idx + 1) % 50 == 0:
            elapsed_time = (datetime.now() - start_time).total_seconds() / 60
            print(f"   Progress: {idx + 1}/{len(movies_to_query)} - {elapsed_time:.1f} min")
        
        # Query OMDb
        info = fetch_omdb_rating(title_norm, api_key)
        
        # Script path
        script_path = os.path.join(scripts_folder, original_file)
        
        # Save result
        new_results.append({
            'index': movie_idx,
            'script_filename': original_file,
            'script_path': script_path,
            'normalized_title': title_norm,
            'omdb_title': info['Title'],
            'omdb_year': info['Year'],
            'omdb_rating_imdb': info['imdbRating'],
            'omdb_rating_rottenomatoes': info['RottenTomatoes'],
            'omdb_plot': info['Plot'],
            'omdb_director': info['Director'],
            'omdb_actors': info['Actors'],
            'omdb_genre': info['Genre'],
            'omdb_type': info['Type'],
            'omdb_response': info['Response'],
            'timestamp': datetime.now().isoformat()
        })
        
        # Small delay
        time.sleep(0.1)
    
    total_time = (datetime.now() - start_time).total_seconds() / 60
    
    # ─────────────────────────────────────────────────────────────────────
    # 6. SAVE NEW RESULTS AND UPDATE CHECKPOINT
    # ─────────────────────────────────────────────────────────────────────
    
    df_new = pd.DataFrame(new_results)
    
    # Combine with previous results
    if df_prev_results is not None:
        df_omdb = pd.concat([df_prev_results, df_new], ignore_index=True)
    else:
        df_omdb = df_new
    
    # Save to temporary file
    df_omdb.to_csv(RESULT_CSV, index=False)
    
    # Update checkpoint
    checkpoint['queries_today'] += len(new_results)
    checkpoint['last_index'] = movies_to_query[-1] if movies_to_query else checkpoint['last_index']
    save_checkpoint(checkpoint)
    
    print(f"\n✅ Queries completed in {total_time:.1f} minutes")
    print(f"   New records: {len(new_results)}")
    print(f"   Total accumulated: {len(df_omdb)}")
    print(f"\n📊 Request usage today:")
    print(f"   Consumed: {checkpoint['queries_today']}/{MAX_REQUESTS_PER_DAY}")
    print(f"   Available tomorrow: {MAX_REQUESTS_PER_DAY}")
    
    # ─────────────────────────────────────────────────────────────────────
    # 7. SHOW PREVIEW
    # ─────────────────────────────────────────────────────────────────────
    
    print("\n" + "="*70)
    print("PREVIEW OF ACCUMULATED DATAFRAME")
    print("="*70)
    print(df_omdb.head())
    
    print("\n" + "="*70)
    print("ACCUMULATED SUMMARY")
    print("="*70)
    print(f"Total records: {len(df_omdb)}")
    print(f"Movies found: {(df_omdb['omdb_response'] == 'Found').sum()}")
    print(f"Not found: {(df_omdb['omdb_response'] != 'Found').sum()}")
    
    if len(pending_movies) > len(movies_to_query):
        print(f"\n⏳ {len(pending_movies) - len(movies_to_query)} movies still pending")
        print(f"   Run this cell tomorrow to continue ✅")


## 4. Clean and Save Results

Load the checkpoint CSV, convert data types (IMDb ratings to `float`, Rotten Tomatoes percentages to `float` in [0, 1]), and save a clean version once all movies have been queried.

In [ ]:
# CELL 4: Save final dataframe and create optimized structure for ML

# Check if the temporary file exists
if os.path.exists("movie_metadata_temp.csv"):
    df_omdb = pd.read_csv("movie_metadata_temp.csv")
    print(f"✅ Dataframe loaded from checkpoint: {len(df_omdb)} records")
else:
    print("⚠️  Temporary file not found. Make sure to run Cell 3 first.")
    df_omdb = None

if df_omdb is not None:
    # A. Save TEMPORARY CSV (keeps the checkpoint)
    print(f"\n📝 Temporary file (checkpoint): movie_metadata_temp.csv")
    
    # B. Create clean version (only found movies)
    df_found = df_omdb[df_omdb['omdb_response'] == 'Found'].copy()
    print(f"\n✅ Movies found: {len(df_found)} of {len(df_omdb)}")
    
    # Convert ratings to numeric
    df_found['omdb_rating_imdb'] = pd.to_numeric(df_found['omdb_rating_imdb'], errors='coerce')
    
    # Rotten Tomatoes cleanup
    df_found['omdb_rating_rottenomatoes'] = df_found['omdb_rating_rottenomatoes'].apply(
        lambda x: float(x.rstrip('%')) / 100 if isinstance(x, str) and x != 'N/A' else None
    )
    df_found['omdb_year'] = pd.to_numeric(df_found['omdb_year'], errors='coerce')
    
    # C. Save clean version ONLY when all have been queried
    if len(df_omdb) == 1298:  # If all movies have been processed
        clean_csv = "movie_metadata_clean.csv"
        df_found.to_csv(clean_csv, index=False)
        print(f"✅ Clean metadata (FINAL) saved to: {clean_csv}")
    else:
        print(f"⏳ Queried: {len(df_omdb)}/1298. Final metadata will be saved when all are done.")
    
    # D. Show statistics
    print("\n" + "="*70)
    print("RATING STATISTICS (Found movies)")
    print("="*70)
    
    imdb_ratings = df_found['omdb_rating_imdb'].dropna()
    rt_ratings = df_found['omdb_rating_rottenomatoes'].dropna()
    
    if len(imdb_ratings) > 0:
        print(f"IMDb Rating:")
        print(f"  Average: {imdb_ratings.mean():.2f}")
        print(f"  Min: {imdb_ratings.min():.2f}, Max: {imdb_ratings.max():.2f}")
        print(f"  Movies with rating: {len(imdb_ratings)}/{len(df_found)}")
    
    if len(rt_ratings) > 0:
        print(f"\nRotten Tomatoes Rating:")
        print(f"  Average: {rt_ratings.mean():.2%}")
        print(f"  Movies with rating: {len(rt_ratings)}/{len(df_found)}")
    
    print("\n" + "="*70)
    print("MACHINE LEARNING RECOMMENDATION")
    print("="*70)
    print("""
✅ RECOMMENDED STRUCTURE (Scalable and efficient)
────────────────────────────────────────────────────
Files created:

1. movie_metadata_temp.csv (CHECKPOINT - in progress)
   - Updated each day with new results
   - Do NOT use for ML yet

2. movie_metadata_clean.csv (FINAL - when all are done)
   - Created when all 1298 movies have been queried
   - Contains: title, year, ratings, genre, directors
   - Use this for ML

3. todos_los_scripts/ (Scripts folder)
   - 1298 .txt files with movie content
   - Referenced by 'script_path' in CSV

Advantages:
  ✓ Separate metadata from scripts (memory-efficient)
  ✓ Load data in batches during training
  ✓ Scalable to millions of movies
  ✓ Ideal for deep learning (LSTM, Transformers, etc.)

Next steps:
  1. Run Cell 3 each day until all 1298 are done
  2. When you reach 1298, you will have movie_metadata_clean.csv
  3. Use that CSV + scripts to train a model
    """)


## 5. Dataset Overview and `MovieDataset` Class

Show rating statistics, display a sample of the collected data, and define `MovieDataset` — a helper class for loading scripts alongside their metadata in batches (useful for model training).

In [ ]:
# CELL 5: Load and display OMDb data (movie_metadata)
import pandas as pd

# Prefer the clean file if it exists, otherwise use the temporary one
if os.path.exists("movie_metadata_clean.csv"):
    df_metadata = pd.read_csv("movie_metadata_clean.csv")
    file_used = "movie_metadata_clean.csv"
elif os.path.exists("movie_metadata_temp.csv"):
    df_metadata = pd.read_csv("movie_metadata_temp.csv")
    file_used = "movie_metadata_temp.csv"
else:
    print("⚠️  No metadata file found. Run Cell 3 first.")
    df_metadata = None

if df_metadata is not None:
    print("="*70)
    print(f"OMDB DATASET: {file_used}")
    print("="*70)
    print(f"\n✅ Dataset loaded: {len(df_metadata)} records")
    print(f"📊 Available columns: {list(df_metadata.columns)}")
    
    print("\n" + "="*70)
    print("DATAFRAME SAMPLE - First 10 rows")
    print("="*70)
    print(df_metadata.head(10))
    
    print("\n" + "="*70)
    print("DETAILED INFO")
    print("="*70)
    print(df_metadata.info())
    
    print("\n" + "="*70)
    print("RATING STATISTICS")
    print("="*70)
    # Convert ratings to numeric if not already
    if 'omdb_rating_imdb' in df_metadata.columns:
        df_metadata['omdb_rating_imdb_numeric'] = pd.to_numeric(df_metadata['omdb_rating_imdb'], errors='coerce')
        ratings_imdb = df_metadata['omdb_rating_imdb_numeric'].dropna()
        if len(ratings_imdb) > 0:
            print(f"\nIMDb Rating:")
            print(f"  Average: {ratings_imdb.mean():.2f}")
            print(f"  Standard deviation: {ratings_imdb.std():.2f}")
            print(f"  Min: {ratings_imdb.min():.2f}, Max: {ratings_imdb.max():.2f}")
            print(f"  Movies with rating: {len(ratings_imdb)}/{len(df_metadata)}")
    
    if 'omdb_rating_rottenomatoes' in df_metadata.columns:
        print(f"\nRotten Tomatoes Rating:")
        print(f"  Movies with rating: {df_metadata['omdb_rating_rottenomatoes'].notna().sum()}/{len(df_metadata)}")
    
    print("\n" + "="*70)
    print("RESPONSE ANALYSIS (Found vs not found)")
    print("="*70)
    if 'omdb_response' in df_metadata.columns:
        print(df_metadata['omdb_response'].value_counts())
    
    print("\n" + "="*70)
    print("SAMPLE OF FOUND MOVIES")
    print("="*70)
    if 'omdb_response' in df_metadata.columns:
        df_found = df_metadata[df_metadata['omdb_response'] == 'Found']
        print(f"Total found: {len(df_found)}")
        print("\nFirst 5 found movies:")
        cols_to_show = ['script_filename', 'omdb_title', 'omdb_year', 'omdb_rating_imdb', 'omdb_genre']
        existing_cols = [col for col in cols_to_show if col in df_found.columns]
        print(df_found[existing_cols].head(5))

# ─────────────────────────────────────────────────────────────────────
# Class to load complete movies (metadata + scripts)
# ─────────────────────────────────────────────────────────────────────

class MovieDataset:
    """Efficient loader for movie data + scripts"""
    
    def __init__(self, csv_path="movie_metadata_clean.csv"):
        self.df = pd.read_csv(csv_path)
        print(f"\n✅ Metadata loaded: {len(self.df)} movies")
    
    def load_script(self, idx):
        """Load script by index"""
        row = self.df.iloc[idx]
        try:
            with open(row['script_path'], 'r', encoding='utf-8') as f:
                return f.read()
        except:
            return None
    
    def get_batch(self, indices, load_scripts=True):
        """Get a batch of movies with or without scripts"""
        batch = self.df.iloc[indices].copy()
        
        if load_scripts:
            batch['script_text'] = [self.load_script(i) for i in indices]
        
        return batch
    
    def get_by_rating_range(self, min_rating=5, max_rating=10):
        """Filter by IMDb rating"""
        mask = (self.df['omdb_rating_imdb'] >= min_rating) & \
               (self.df['omdb_rating_imdb'] <= max_rating)
        return self.df[mask]

print("\n" + "="*70)
print("NEXT STEPS")
print("="*70)
print("""
✅ Now that you have OMDb metadata, you can:

1. Keep running Cell 3 each day until all 1298 movies are done

2. Once complete, merge with the Oscar dataset:
   dataset = MovieDataset("movie_metadata_clean.csv")
   df_oscar = pd.read_csv("full_data.csv", delimiter='\\t')
   df_merged = df_oscar.merge(dataset.df, 
                              left_on='Film',
                              right_on='omdb_title',
                              how='left')

3. Use df_merged to train your ML model:
   - Features: Script content (script_path)
   - Target: omdb_rating_imdb
""")
